<a href="https://colab.research.google.com/github/TinkerTechie/FakeNewsDetector/blob/main/model1_for_fake_news_generator_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd

df = pd.read_csv('politifact_final_dataset (1) - politifact_final_dataset (1).csv.csv')
df.head()

,id,text,label,url,source,fetch_status
0,politifact7714,NaN,0,http://www.cq.com/doc/newsmakertranscripts-426...,politifact_real,low_content
1,politifact14818,NaN,1,thehill.com/policy/national-security/355749-fb...,politifact_fake,failed
2,politifact15270,NaN,1,http://usanews24h.tk/index.php/2018/04/12/obam...,politifact_fake,failed
3,politifact15292,Submitted by MAGA Student Posted 2 days ago ...,1,https://web.archive.org/web/20180424000839/htt...,politifact_fake,success
4,politifact2093,"Transcript: Biden July 11, 2010 &#151; --JAKE ...",0,http://abcnews.go.com/ThisWeek/week-transcript...,politifact_real,success


In [ ]:
!pip install requests beautifulsoup4 playwright pandas
!playwright install

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import signal
import os

# =========================
# PATHS
# =========================
INPUT_CSV = "/content/politifact_final_dataset (1) - politifact_final_dataset (1).csv.csv"
OUTPUT_CSV = "politifact_with_extracted_text.csv"

# =========================
# SETTINGS
# =========================
MIN_TEXT_LENGTH = 300
REQUEST_TIMEOUT = 5
READER_TIMEOUT = 8
ROW_TIMEOUT = 12   # max seconds per URL

BLOCKED_DOMAINS = [
    "linkedin.com","instagram.com","facebook.com",
    "twitter.com","x.com","docs.google.com","drive.google.com"
]

# =========================
# TIMEOUT HANDLER
# =========================
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

signal.signal(signal.SIGALRM, timeout_handler)

# =========================
# HELPERS
# =========================
def normalize_url(url):
    if pd.isna(url):
        return None
    url = str(url).strip()
    if not url.startswith("http"):
        url = "https://" + url
    return url

def is_blocked(url):
    return any(d in url.lower() for d in BLOCKED_DOMAINS)

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script","style","noscript","header","footer","nav"]):
        tag.extract()
    return soup.get_text(" ", strip=True)

# =========================
# EXTRACTION METHODS
# =========================
def try_requests(url):
    try:
        r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=REQUEST_TIMEOUT)
        if r.status_code != 200:
            return None
        text = clean_html(r.text)
        if len(text) < MIN_TEXT_LENGTH:
            return None
        return text
    except:
        return None

def try_reader(url):
    try:
        r = requests.get(f"https://r.jina.ai/{url}", timeout=READER_TIMEOUT)
        if r.status_code != 200:
            return None
        text = r.text.strip()
        if len(text) < MIN_TEXT_LENGTH:
            return None
        return text
    except:
        return None

def extract_from_url(url):
    if not url or is_blocked(url):
        return None

    text = try_requests(url)
    if text:
        return text

    text = try_reader(url)
    if text:
        return text

    return None

# =========================
# SAFE WRAPPER WITH TIMEOUT
# =========================
def extract_with_timeout(url):
    try:
        signal.alarm(ROW_TIMEOUT)
        text = extract_from_url(url)
        signal.alarm(0)
        return text
    except TimeoutException:
        return None

# =========================
# LOAD DATASET
# =========================
df = pd.read_csv(INPUT_CSV)

extracted = []

# =========================
# MAIN LOOP
# =========================
for i, row in df.iterrows():
    print("Processing row:", i)

    # 1️⃣ dataset already has text
    if pd.notna(row.get("text")) and len(str(row["text"])) > MIN_TEXT_LENGTH:
        extracted.append(row["text"])
        continue

    # 2️⃣ skip known failed URLs
    if str(row.get("fetch_status")).lower() == "failed":
        extracted.append(None)
        continue

    # 3️⃣ normalize URL
    url = normalize_url(row.get("url"))

    # 4️⃣ extract safely
    article = extract_with_timeout(url)
    extracted.append(article)

# =========================
# SAVE RESULT
# =========================
df["extracted_text"] = extracted
output_dir = os.path.dirname(OUTPUT_CSV)
if output_dir: # Only create directory if it's not an empty string
    os.makedirs(output_dir, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
from google.colab import files
files.download("politifact_with_extracted_text.csv")

print("✅ DONE — saved to:", OUTPUT_CSV)

In [ ]:
df["extracted_text"].isnull().sum()

In [ ]:
final_text = df.loc[row.name, "extracted_text"]

In [ ]:
final_text = row["text"]
final_text

In [ ]:
df = df.copy()

df["final_text"] = df["extracted_text"].fillna(df["text"])
df["final_text"] = df["final_text"].fillna("").astype(str)

df["url"] = df["url"].fillna("").astype(str)
df["label"] = df["label"].astype(int)

**Matrix 1 (Linguistic Style)**

Install sentiment library

In [ ]:
!pip install vaderSentiment

In [ ]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# =========================
# LEXICONS (edit/expand anytime)
# =========================
SENSATIONAL_WORDS = {
    "shocking","unbelievable","explosive","disturbing",
    "exposed","bombshell","outrageous","terrifying"
}

CLICKBAIT_PHRASES = [
    "you won't believe",
    "they don't want you to know",
    "secret",
    "hidden truth",
    "what happens next",
    "this will shock you"
]

HYPERBOLE_WORDS = {
    "always","never","everyone","everything",
    "guaranteed","forever","completely","absolutely"
}

# =========================
# TOKENIZER
# =========================
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# =========================
# FEATURE FUNCTIONS
# =========================
def sensational_density(text):
    words = tokenize(text)
    if not words:
        return 0
    count = sum(1 for w in words if w in SENSATIONAL_WORDS)
    return count / len(words)

def clickbait_density(text):
    text_lower = text.lower()
    word_count = max(len(tokenize(text)), 1)
    count = sum(text_lower.count(p) for p in CLICKBAIT_PHRASES)
    return count / word_count

def hyperbole_density(text):
    words = tokenize(text)
    if not words:
        return 0
    count = sum(1 for w in words if w in HYPERBOLE_WORDS)
    return count / len(words)

def caps_punct_intensity(text):
    words = text.split()
    if not words:
        return 0

    caps = sum(1 for w in words if w.isupper() and len(w) > 2)
    exclam = text.count("!")
    ques = text.count("?")

    return (caps + exclam + ques) / len(words)

def emotional_intensity(text):
    score = analyzer.polarity_scores(text)
    return abs(score["compound"])

# =========================
# APPLY MATRIX 1
# =========================
df["sensational_density"] = df["final_text"].apply(sensational_density)
df["clickbait_density"] = df["final_text"].apply(clickbait_density)
df["hyperbole_density"] = df["final_text"].apply(hyperbole_density)
df["caps_punct_intensity"] = df["final_text"].apply(caps_punct_intensity)
df["emotional_intensity"] = df["final_text"].apply(emotional_intensity)

print("✅ Matrix 1 features created")

In [ ]:
df[[
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity"
]].describe()

In [ ]:
df.to_csv("/mnt/data/politifact_matrix1.csv", index=False)

**Matrix 2**

Install embedding model

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

model = SentenceTransformer("all-MiniLM-L6-v2")

# =========================
# Helpers
# =========================
def split_headline_body(text):
    """
    Approximate headline as first sentence.
    Body = remaining text.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if len(sentences) == 0:
        return "", ""
    if len(sentences) == 1:
        return sentences[0], sentences[0]
    return sentences[0], " ".join(sentences[1:])

def paragraph_split(text):
    """
    Split into pseudo-paragraphs.
    """
    paras = re.split(r"\n+|\.\s+", text)
    paras = [p.strip() for p in paras if len(p.strip()) > 40]
    return paras


# =========================
# Feature 1 — Headline-Body Similarity
# =========================
def headline_body_similarity(text):
    if not text or len(text) < 50:
        return 0

    headline, body = split_headline_body(text)

    emb = model.encode([headline, body])
    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]

    return float(sim)


# =========================
# Feature 2 — Topic Drift
# =========================
def topic_drift(text):
    paras = paragraph_split(text)

    if len(paras) < 2:
        return 0

    emb = model.encode(paras)

    sims = []
    for i in range(len(emb) - 1):
        s = cosine_similarity([emb[i]], [emb[i+1]])[0][0]
        sims.append(s)

    # drift = variation in similarity
    return float(np.std(sims))


# =========================
# Apply Matrix 2
# =========================
df["headline_body_similarity"] = df["final_text"].apply(headline_body_similarity)
df["topic_drift"] = df["final_text"].apply(topic_drift)

print("✅ Matrix 2 features created")

In [ ]:
df[["headline_body_similarity","topic_drift"]].describe()

In [ ]:
df.to_csv("/mnt/data/politifact_matrix2.csv", index=False)

**Matrix 3**

In [ ]:
from urllib.parse import urlparse
import numpy as np

# =========================
# Extract domain from URL
# =========================
def extract_domain(url):
    if not isinstance(url, str) or url.strip() == "":
        return "unknown"
    try:
        domain = urlparse(url).netloc.lower()
        if domain.startswith("www."):
            domain = domain[4:]
        return domain if domain else "unknown"
    except:
        return "unknown"

df["domain"] = df["url"].apply(extract_domain)

# =========================
# Compute fake ratio per domain
# =========================
domain_stats = df.groupby("domain")["label"].agg(["count","mean"]).reset_index()

domain_stats.columns = ["domain","article_count","fake_ratio"]

# =========================
# Convert to reliability
# =========================
domain_stats["source_reliability"] = 1 - domain_stats["fake_ratio"]

# =========================
# Merge back to dataset
# =========================
df = df.merge(
    domain_stats[["domain","source_reliability"]],
    on="domain",
    how="left"
)

# =========================
# Handle unseen / rare domains
# =========================
MIN_DOMAIN_ARTICLES = 3

rare_domains = domain_stats[
    domain_stats["article_count"] < MIN_DOMAIN_ARTICLES
]["domain"]

df.loc[df["domain"].isin(rare_domains), "source_reliability"] = 0.5
df["source_reliability"] = df["source_reliability"].fillna(0.5)

print("✅ Matrix 3 feature created")

In [ ]:
df["source_reliability"].describe()

In [ ]:
df.to_csv("/mnt/data/politifact_matrix3.csv", index=False)

In [ ]:
features = [
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity",
    "headline_body_similarity",
    "topic_drift",
    "source_reliability"
]

X = df[features]
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
features = [
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity",
    "headline_body_similarity",
    "topic_drift",
    "source_reliability"
]

X = df[features]
y = df["label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

importance = pd.Series(
    model.coef_[0],
    index=features
).sort_values(ascending=False)

print("Feature Importance:")
print(importance)

In [ ]:
df["fake_probability"] = model.predict_proba(
    scaler.transform(df[features])
)[:,1]

In [ ]:
df.to_csv("/mnt/data/politifact_with_matrices_and_model.csv", index=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Results")
print("Accuracy :", acc_rf)
print("Precision:", prec_rf)
print("Recall   :", rec_rf)
print("F1 Score :", f1_rf)

In [ ]:
rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(rf_importance)